# Centroid-Based Binary Classifier for P4 IDS
Upload your dataset and compute centroids for rules.json

In [ ]:
# Step 1: Mount Google Drive and load dataset from shared folder
# This ensures all computation runs on Colab's CPUs, not your local device
from google.colab import drive
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import LabelEncoder
import os

# Mount Google Drive
print('Mounting Google Drive...')
drive.mount('/content/drive')

# Access the shared folder using its ID
FOLDER_ID = '1f_C18E3h5-_RUT0tTnKTAXhYxyTmdO_H'
folder_path = f'/content/drive/.shortcut-targets-by-id/{FOLDER_ID}'

print(f'Accessing shared folder: {folder_path}')

# List files in the folder
try:
    files = os.listdir(folder_path)
    print(f'Files in shared folder: {files}')
    csv_files = [f for f in files if f.endswith('.csv')]
    print(f'CSV files found: {csv_files}')
except Exception as e:
    print(f'Error accessing folder: {e}')
    print('Make sure the folder is shared with public access and you have permission.')
    df = None

# Load the dataset
if csv_files:
    # Based on your code, try Data.csv and Label.csv
    try:
        data_df = pd.read_csv(os.path.join(folder_path, 'Data.csv'))
        label_df = pd.read_csv(os.path.join(folder_path, 'Label.csv'))
        df = pd.concat([data_df, label_df], axis=1)
        print('✓ Successfully loaded Data.csv and Label.csv from Drive')
    except Exception as e:
        print(f'Error loading Data.csv/Label.csv: {e}')
        # Try loading the first CSV
        df = pd.read_csv(os.path.join(folder_path, csv_files[0]))
        print(f'Loaded {csv_files[0]} instead')
else:
    print('No CSV files found in the shared folder.')
    df = None

if df is not None:
    print(f'\nDataset shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()[:10]}...')  # Show first 10 columns
else:
    print('ERROR: Could not load dataset from Drive.')

ModuleNotFoundError: No module named 'google'

## How to get your Google Drive FILE_ID:
1. Upload your CSV file to Google Drive (free account, unlimited storage for learning)
2. Right-click the file → **Share**
3. Click "Change to anyone with the link" (or keep as is)
4. Copy the shareable link, which looks like:
   ```
   https://drive.google.com/file/d/1a2b3c4d5e6f7g8h9i0j/view?usp=sharing
   ```
5. Extract the FILE_ID (the long string between `/d/` and `/`):
   ```
   FILE_ID = 1a2b3c4d5e6f7g8h9i0j
   ```
6. Paste it in the cell above where it says `YOUR_FILE_ID_HERE`

This way, your huge dataset stays on Google Drive and you download it on-demand!

In [ ]:
# Step 2: Inspect the loaded dataset
if df is not None:
    print(f'Dataset shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    print(f'\nFirst few rows:')
    print(df.head())
else:
    print('ERROR: Dataset not loaded. Check your FILE_ID or URL and try again.')

In [ ]:
# Step 3: Preprocess data
# Assuming 'Label' column exists (e.g., 'BENIGN' for normal, attack names for malicious)
# If your label column has a different name, update below

label_column = 'Label'  # Change this if your column is named differently

# Map labels to binary: 0 = normal (BENIGN), 1 = attack
label_encoder = LabelEncoder()
df[label_column] = label_encoder.fit_transform(df[label_column])

print(f'Label mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}')

# Select numeric features (exclude label)
features = [col for col in df.columns if col != label_column and pd.api.types.is_numeric_dtype(df[col])]
X = df[features]
y = df[label_column]

print(f'Number of features: {len(features)}')
print(f'Feature names: {features[:10]}...')  # Show first 10

In [ ]:
# Step 4: Binarize features (if not already binary)
# Threshold at median for each feature
X_binary = X.copy()
for col in X_binary.columns:
    median_val = X_binary[col].median()
    X_binary[col] = (X_binary[col] > median_val).astype(int)

print('Features binarized at median')
print(f'Binarized data (first 5 rows):')
print(X_binary.head())

In [ ]:
# Step 5: Compute centroids (majority vote per feature bit)
centroids = {}
class_names = {0: 'NORMAL', 1: 'ATTACK'}

for class_label in [0, 1]:
    class_data = X_binary[y == class_label]
    centroid = class_data.mean(axis=0).round().astype(int).tolist()  # Majority vote
    centroids[f'class_{class_label}'] = centroid
    print(f'{class_names[class_label]} Centroid (first 20 bits): {centroid[:20]}...')
    print(f'  Centroid length: {len(centroid)} bits')

In [ ]:
# Step 6: Generate rules.json format for P4
rules_output = []
feature_list = features  # Store feature order for reference

for class_name, centroid in centroids.items():
    rule = {
        "table": "decision_table",
        "centroid": centroid,
        "action": "drop" if "1" in class_name else "forward",
        "class": class_name
    }
    rules_output.append(rule)

# Also save feature order for P4 reference
output = {
    "features": feature_list,
    "rules": rules_output,
    "metadata": {
        "classifier": "centroid_binary",
        "num_features": len(feature_list),
        "class_mapping": {"0": "NORMAL", "1": "ATTACK"}
    }
}

print('Rules generated:')
print(json.dumps(output, indent=2))

In [ ]:
# Step 7: Save to centroids_rules.json in local directory
output_filename = 'centroids_rules.json'
with open(output_filename, 'w') as f:
    json.dump(output, f, indent=4)

print(f'Saved to {output_filename}')
print(f'File location: {os.path.abspath(output_filename)}')
print(f'\nYou can now use centroids_rules.json to replace rules.json in your P4 project')